# TFM — WESAD Deep Learning Baselines Re-evaluation (20s Windows)
## Version 6: Baselines Comparison (BiLSTM, CNN1D, CNN1D-MS, Hybrid MST-Conformer)

### Addressing Reviewer Feedback for MDPI Sensors (Manuscript `sensors-4523028`)


In [30]:
# ==== BLOCK 1: SETUP & DEPENDENCIES ====
from google.colab import drive
import time, os, re, zipfile, fnmatch, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as sig
from scipy import stats as sp_stats
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
)

# Mount Google Drive if in Colab
if os.path.isdir('/content'):
    drive.mount('/content/drive')

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q scipy scikit-learn matplotlib pandas tqdm optuna wandb seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Dedicated Target Output Directory for v6 Baselines
# WORKING_DIR path should be modified so that it matches the google drive path where the whole folder has been copied to.
WORKING_DIR = "/content/drive/MyDrive/WESAD_Benchmark_DL"
TARGET_DIR = WORKING_DIR
os.makedirs(TARGET_DIR, exist_ok=True)

print('=' * 75)
print('WESAD Deep Learning Baselines Evaluation (20-second Windows)')
print(f'PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
print(f'Target Output Directory: {TARGET_DIR}')
print('=' * 75)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WESAD Deep Learning Baselines Evaluation (20-second Windows)
PyTorch Version: 2.11.0+cu128 | CUDA Available: True
Active GPU: Tesla T4
Target Output Directory: /content


In [24]:
###### BLOCK 2: LOAD MATLAB TABLES   ######

from google.colab import drive
import h5py
import numpy as np
import os # Import os for path joining
import pandas as pd

# Ensure Google Drive is mounted, as connection can sometimes be lost.
# This check prevents re-mounting if already mounted and re-establishes connection if lost.
if not os.path.exists('/content/drive/MyDrive'):
    print("Google Drive not mounted, attempting to mount...")
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

# ============================================================
# FEATURES FILE
# ============================================================

# Use the full path for the CSV file
# This file has to be loaded as the window labels for the raw signals are taken from the feature file.
df_feat = pd.read_csv(os.path.join(WORKING_DIR, 'All_ECR_9.csv'))

# ============================================================
# SIGNALS FILE
# ============================================================

signals_mat = h5py.File(os.path.join(WORKING_DIR,'PER_SUBEJCT_WESAD_raw_signals_norm_allSubjects.mat'),'r')

print("\n")
print("="*60)
print("SIGNALS FILE")
print("="*60)

print("Keys:")
print(list(signals_mat.keys()))

for k in signals_mat.keys():
    try:
        print(f"{k}: shape={signals_mat[k].shape}, dtype={signals_mat[k].dtype}")
    except:
        print(k)

# ============================================================
# DETAILS OF SUBJECT
# ============================================================

if 'Subject' in signals_mat:

    print("\n")
    print("="*60)
    print("DETAILS OF Subject")
    print("="*60)

    print(signals_mat['Subject'])
    print("shape =", signals_mat['Subject'].shape)
    print("dtype =", signals_mat['Subject'].dtype)

    try:
        print("raw values:")
        print(np.array(signals_mat['Subject']))
    except Exception as e:
        print(e)

# ============================================================
# DETAILS OF X_raw_norm
# ============================================================

if 'X_raw_norm' in signals_mat:

    print("\n")
    print("="*60)
    print("DETAILS OF X_raw_norm")
    print("="*60)

    Xtmp = np.array(signals_mat['X_raw_norm'])

    print("shape =", Xtmp.shape)
    print("dtype =", Xtmp.dtype)

# ============================================================
# DETAILS OF y
# ============================================================

if 'y' in signals_mat:

    print("\n")
    print("="*60)
    print("DETAILS OF y")
    print("="*60)

    ytmp = np.array(signals_mat['y'])

    print("shape =", ytmp.shape)
    print("dtype =", ytmp.dtype)

    try:
        print("unique values:", np.unique(ytmp))
    except:
        pass

#### SIGNAL REFORMAT AND DATASET CONSISTENCY CHECKS

import numpy as np

X_raw_norm = np.array(signals_mat['X_raw_norm'])
X_raw_norm = np.transpose(X_raw_norm, (2,1,0))

print(X_raw_norm.shape)
print(df_feat.shape)

SUBJECTS = ['S2', 'S3', 'S4', 'S5', 'S6','S7', 'S8', 'S9', 'S10', 'S11',
            'S13', 'S14', 'S15', 'S16', 'S17']
subject_data = {}
subject_data_norm = {}

for sid in SUBJECTS:
    sid_sync = f"{sid}_sync"
    idx = (df_feat['Subject'] == sid_sync)
    y_sid = (df_feat.loc[idx, 'y'].values.astype(np.int64))
    X_raw_sid = X_raw_norm[idx.values]

    subject_data[sid] = (X_raw_sid,y_sid);
    subject_data_norm[sid] = (X_raw_sid,y_sid)

    print(f"{sid}: "f"{len(y_sid)} windows | "f"X_raw={X_raw_sid.shape} | ")
print(subject_data_norm.keys())

xr, y = subject_data_norm['S2']

print(xr.shape)
print(y.shape)

# ==== GLOBAL CONFIG ====

FS = 700
WINDOW_SEC = 20
WINDOW_SIZE = WINDOW_SEC * FS
STRIDE = WINDOW_SIZE // 4
NUM_CHANNELS = 2
LABEL_MAP = {1: 0, 2: 1, 3: 1, 4: 0}
RANDOM_STATE = 42

#### DATASET CONSISTENCY CHECK

total_windows = 0
for sid in SUBJECTS:
    total_windows += len(subject_data_norm[sid][1])

print(total_windows)
print('Subject Class0 Class1')
for sid in SUBJECTS:
    y = subject_data_norm[sid][1]
    print(sid,np.sum(y == 0),np.sum(y == 1))

for sid in SUBJECTS:
    xr, y = subject_data_norm[sid]
    assert xr.shape[2] == 2

print("Dataset check passed.")

Google Drive is already mounted.


SIGNALS FILE
Keys:
['#refs#', '#subsystem#', 'Subject', 'X_raw_norm', 'y']
#refs#
#subsystem#
Subject: shape=(1, 6), dtype=uint32
X_raw_norm: shape=(2, 14000, 8991), dtype=float32
y: shape=(1, 8991), dtype=float64


DETAILS OF Subject
<HDF5 dataset "Subject": shape (1, 6), type "<u4">
shape = (1, 6)
dtype = uint32
raw values:
[[3707764736          2          1          1          1          1]]


DETAILS OF X_raw_norm
shape = (2, 14000, 8991)
dtype = float32


DETAILS OF y
shape = (1, 8991)
dtype = float64
unique values: [0. 1.]
(8991, 14000, 2)
(8991, 11)
(8991, 14000, 2)
S2: 578 windows | X_raw=(578, 14000, 2) | 
S3: 587 windows | X_raw=(587, 14000, 2) | 
S4: 592 windows | X_raw=(592, 14000, 2) | 
S5: 602 windows | X_raw=(602, 14000, 2) | 
S6: 598 windows | X_raw=(598, 14000, 2) | 
S7: 597 windows | X_raw=(597, 14000, 2) | 
S8: 601 windows | X_raw=(601, 14000, 2) | 
S9: 599 windows | X_raw=(599, 14000, 2) | 
S10: 615 windows | X_raw=(615, 14000, 2) 

In [35]:
# ==== BLOCK 4: PYTORCH DATASET & LOSO CROSS-VALIDATION SPLITS ====
class WESADDataset20s(Dataset):
    def __init__(self, X_raw, y, augment=False):
        self.X_raw = torch.FloatTensor(X_raw).permute(0, 2, 1)  # (N, C, T=14000)
        self.y = torch.FloatTensor(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_raw = self.X_raw[idx].clone()
        if self.augment:
            x_raw = self._augment_raw(x_raw)
        return x_raw, self.y[idx]

    def _augment_raw(self, x):
        C, T = x.shape
        if torch.rand(1).item() < 0.7:
            x = x + torch.randn_like(x) * 0.02
        if torch.rand(1).item() < 0.7:
            scale = 0.8 + torch.rand(1).item() * 0.4
            x = x * scale
        if torch.rand(1).item() < 0.5:
            shift = int(torch.randint(-T // 10, T // 10 + 1, (1,)).item())
            x = torch.roll(x, shifts=shift, dims=-1)
        if torch.rand(1).item() < 0.5:
            mask_len = int(torch.randint(T // 20, T // 5, (1,)).item())
            mask_start = int(torch.randint(0, T - mask_len, (1,)).item())
            x[:, mask_start:mask_start + mask_len] = 0.0
        return x

def get_loso_splits_20s(subject_data_norm, test_sid, val_sid=None):
    '''Builds exact Train / Validation / Test partitions for 15-fold LOSO.
    Identical to the partition schedule used in the Ablation Study (v5 Ablation v2).'''
    sids = list(subject_data_norm.keys())
    ts_Xr, ts_y = subject_data_norm[test_sid]
    remaining = [s for s in sids if s != test_sid]
    if val_sid is None:
        test_idx = sids.index(test_sid)
        val_sid = remaining[test_idx % len(remaining)]
    vl_Xr, vl_y = subject_data_norm[val_sid]
    train_sids = [s for s in remaining if s != val_sid]
    tr_Xr = np.vstack([subject_data_norm[s][0] for s in train_sids])
    tr_y = np.concatenate([subject_data_norm[s][1] for s in train_sids])
    return tr_Xr, tr_y, vl_Xr, vl_y, ts_Xr, ts_y, val_sid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Computing Device: {device}')


Active Computing Device: cuda


In [36]:
# ==== BLOCK 5: ALL 3 DEEP LEARNING ARCHITECTURES (ADAPTED FOR 20s WINDOWS) ====
# 1. BiLSTM Baseline
class BiLSTM(nn.Module):
    def __init__(self, in_ch=2, dropout=0.3, **kwargs):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.lstm1 = nn.LSTM(64, 32, batch_first=True, bidirectional=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(64, 16, batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x_raw):
        x = self.conv(x_raw)       # (B, 64, T/4)
        x = x.permute(0, 2, 1)     # (B, T/4, 64)
        x, _ = self.lstm1(x)
        x = self.drop1(x)
        x, _ = self.lstm2(x)
        x = self.drop2(x)
        x = x.mean(dim=1)          # GlobalAveragePooling1D -> (B, 32)
        return self.head(x).squeeze(-1)

# 2. CNN1D Baseline
class CNN1D(nn.Module):
    def __init__(self, in_ch=2, dropout=0.5, **kwargs):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_ch, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, x_raw):
        x = self.features(x_raw)
        return self.head(x).squeeze(-1)

# 3. CNN1D-MS Baseline
class CNN1D_MS(nn.Module):
    def __init__(self, in_ch=2, dropout=0.5, **kwargs):
        super().__init__()
        def make_branch(ks):
            pad = ks // 2
            return nn.Sequential(
                nn.Conv1d(in_ch, 32, ks, padding=pad), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(32, 64, ks, padding=pad), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(64, 128, ks, padding=pad), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
                nn.AdaptiveAvgPool1d(1), nn.Flatten()
            )
        self.branch1 = make_branch(3)
        self.branch2 = make_branch(7)
        self.branch3 = make_branch(15)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(384, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, x_raw):
        b1 = self.branch1(x_raw)
        b2 = self.branch2(x_raw)
        b3 = self.branch3(x_raw)
        fused = torch.cat([b1, b2, b3], dim=-1)  # (B, 384)
        return self.head(fused).squeeze(-1)

BASELINES_MODELS = {'BiLSTM': BiLSTM,'CNN1D': CNN1D,'CNN1D-MS': CNN1D_MS}

print('=' * 80)
print(f'{"Model Key":<25s} | {"Class Name":<22s} | {"Parameters":>10s} | {"Output Shape":>14s}')
print('=' * 80)
_xr = torch.randn(4, NUM_CHANNELS, WINDOW_SIZE)
for m_key, m_cls in BASELINES_MODELS.items():
    _m = m_cls(in_ch=NUM_CHANNELS)
    _o = _m(_xr)
    _p = sum(p.numel() for p in _m.parameters())
    print(f'{m_key:<25s} | {m_cls.__name__:<22s} | {_p:>10,d} | {str(list(_o.shape)):>14s}')
    del _m
del _xr
print('=' * 80)
print('All 3 architectures verified successfully on 20s windows (14,000 samples)!')


Model Key                 | Class Name             | Parameters |   Output Shape
BiLSTM                    | BiLSTM                 |     47,521 |            [4]
CNN1D                     | CNN1D                  |     60,513 |            [4]
CNN1D-MS                  | CNN1D_MS               |    284,321 |            [4]
All 3 architectures verified successfully on 20s windows (14,000 samples)!


In [37]:
# ==== BLOCK 6: FOCAL LOSS & METRIC UTILITIES (WITH SPECIFICITY) ====
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, logits, targets):
        if self.ls > 0:
            targets = targets * (1 - self.ls) + 0.5 * self.ls
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.sigmoid(logits)
        pt = targets * pt + (1 - targets) * (1 - pt)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_t * ((1 - pt) ** self.gamma) * bce).mean()

def compute_all_metrics(y_true, y_probs, threshold=0.5):
    y_pred = (y_probs >= threshold).astype(int)
    y_true = y_true.astype(int)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc_val = roc_auc_score(y_true, y_probs)
    except Exception:
        auc_val = 0.5

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / max(tp + fn, 1)  # Sensitivity / Recall
    spec = tn / max(tn + fp, 1)  # Specificity (TN / (TN + FP))

    return {
        'accuracy': acc,
        'bal_acc': bal_acc,
        'f1': f1,
        'auc': auc_val,
        'sensitivity': sens,
        'specificity': spec,
        'cm': cm
    }

def find_optimal_threshold(probs, labels):
    best_t, best_score = 0.5, -1.0
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (probs >= t).astype(int)
        labs = labels.astype(int)
        tp = np.sum((preds == 1) & (labs == 1))
        tn = np.sum((preds == 0) & (labs == 0))
        fp = np.sum((preds == 1) & (labs == 0))
        fn = np.sum((preds == 0) & (labs == 1))
        bal_acc = ((tp / max(tp + fn, 1)) + (tn / max(tn + fp, 1))) / 2.0
        if bal_acc > best_score:
            best_score, best_t = bal_acc, t
    return best_t, best_score

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xr, yb in loader:
        xr = xr.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xr)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    for xr, yb in loader:
        xr = xr.to(device)
        logits = model(xr)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_targets.extend(yb.numpy())
    return np.array(all_probs), np.array(all_targets)


In [38]:
# ==== BLOCK 7: STANDARDIZED 15-FOLD LOSO EVALUATION ENGINE ====
def run_loso_for_model(model_name, ModelClass, model_kwargs=None, n_epochs=35, batch_size=64, lr=1e-3):
    if model_kwargs is None: model_kwargs = {}
    clean_name = model_name.replace(" ", "_").replace("-", "_")
    csv_path = os.path.join(TARGET_DIR, f'{clean_name}_loso_metrics_20s.csv')

    # If already evaluated and saved, load directly to save hours
    if os.path.isfile(csv_path):
        print(f"[INFO] Found existing results for {model_name} at {csv_path}. Loading...")
        df_existing = pd.read_csv(csv_path)
        return df_existing

    print("\n" + "="*85)
    print(f"STARTING 15-FOLD LOSO EVALUATION: {model_name.upper()} (20s Windows)")
    print("Partition Schedule: Identical to Ablation Study (v5 Ablation v2)")
    print("="*85)

    criterion = FocalLoss(alpha=0.5, gamma=2.0, label_smoothing=0.05)
    sids = list(subject_data_norm.keys())
    fold_results = []

    t0_model = time.time()
    for fold_i, test_sid in enumerate(sids):
        t0_fold = time.time()
        remaining = [s for s in sids if s != test_sid]
        val_sid = remaining[fold_i % len(remaining)]  # Exact Ablation partition schedule

        tr_Xr, tr_y, vl_Xr, vl_y, ts_Xr, ts_y, val_sid = \
            get_loso_splits_20s(subject_data_norm, test_sid, val_sid=val_sid)

        train_ds = WESADDataset20s(tr_Xr, tr_y, augment=True)
        val_ds   = WESADDataset20s(vl_Xr, vl_y, augment=False)
        test_ds  = WESADDataset20s(ts_Xr, ts_y, augment=False)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

        torch.manual_seed(RANDOM_STATE + fold_i + 1)
        model = ModelClass(in_ch=NUM_CHANNELS, **model_kwargs).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

        best_val_bal_acc = -1.0
        best_model_weights = None

        for ep in range(1, n_epochs + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            scheduler.step()

            val_probs, val_targets = evaluate_model(model, val_loader)
            opt_thresh, val_bal_acc = find_optimal_threshold(val_probs, val_targets)

            if val_bal_acc > best_val_bal_acc:
                best_val_bal_acc = val_bal_acc
                best_model_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_model_weights)
        val_probs, val_targets = evaluate_model(model, val_loader)
        opt_thresh, _ = find_optimal_threshold(val_probs, val_targets)

        test_probs, test_targets = evaluate_model(model, test_loader)
        m = compute_all_metrics(test_targets, test_probs, threshold=opt_thresh)
        fold_time = time.time() - t0_fold

        row = {
            'model': model_name,
            'fold': fold_i + 1,
            'test_subject': test_sid,
            'val_subject': val_sid,
            'optimal_threshold': round(opt_thresh, 3),
            'bal_acc': round(m['bal_acc'], 4),
            'f1': round(m['f1'], 4),
            'auc': round(m['auc'], 4),
            'accuracy': round(m['accuracy'], 4),
            'sensitivity': round(m['sensitivity'], 4),
            'specificity': round(m['specificity'], 4),
            'runtime_sec': round(fold_time, 2)
        }
        fold_results.append(row)

        print(f"[{fold_i+1:02d}/15] Fold {test_sid:<4s} (Val={val_sid:<4s}) in {fold_time:4.1f}s | "
              f"BalAcc={m['bal_acc']:.3f} | F1={m['f1']:.3f} | AUC={m['auc']:.3f} | "
              f"Sens={m['sensitivity']:.3f} | Spec={m['specificity']:.3f}")

    df_res = pd.DataFrame(fold_results)
    df_res.to_csv(csv_path, index=False)
    print(f"\nSaved 15-fold metrics for {model_name} to: {csv_path}")
    print(f"Total Model Time: {(time.time()-t0_model)/60:.2f} minutes\n")
    return df_res


In [ ]:
# ==== BLOCK 8: EXECUTE LOSO FOR ALL 3 DEEP LEARNING ARCHITECTURES ====
# Enforces exact same partitions and reuses Hybrid MST-Conformer from the Ablation Study

all_results = {}

# 1. Check if Hybrid MST-Conformer results from the Ablation Study already exist
# (to guarantee 100% numerical identity between Table 7 and Table 11 in the manuscript)
candidate_csv = os.path.join(WORKING_DIR, 'loso_results_full.csv')
if os.path.isfile(candidate_csv):
    try:
        df_full = pd.read_csv(candidate_csv)
        if len(df_full) == 15 and 'test_subject' in df_full.columns:
            print(f"[REUSING ABLATION RESULTS] Hybrid MST-Conformer loaded directly from: {candidate_csv}")
            if 'balanced_acc' in df_full.columns and 'bal_acc' not in df_full.columns:
                df_full['bal_acc'] = df_full['balanced_acc']
            if 'threshold' in df_full.columns and 'optimal_threshold' not in df_full.columns:
                df_full['optimal_threshold'] = df_full['threshold']
            df_full['model'] = 'Hybrid MST-Conformer'
            all_results['Hybrid MST-Conformer'] = df_full
            hybrid_found = True
    except Exception:
        pass

# 2. Run the 3 Baseline models using the exact same partitions
for m_name in ['BiLSTM', 'CNN1D', 'CNN1D-MS']:
    m_cls = BASELINES_MODELS[m_name]
    print(f'\nRunning baseline model: {m_name}')
    df_m = run_loso_for_model(m_name, m_cls, n_epochs=35, batch_size=64, lr=1e-3)
    all_results[m_name] = df_m

# 3. If Hybrid MST-Conformer was not loaded send error.
if not hybrid_found:
    raise FileNotFoundError("Hybrid MST-Conformer results (loso_results_full.csv) were not found in any location.")

# Concatenate all fold results into master dataframe
df_master = pd.concat(list(all_results.values()), ignore_index=True)
master_csv = os.path.join(TARGET_DIR, 'loso_all_models_metrics_20s.csv')
df_master.to_csv(master_csv, index=False)
print(f'\n[MASTER COMPLETE] All model fold metrics saved to: {master_csv}')

# Summary Table (Mean ± Std)
model_display_order = ['BiLSTM', 'CNN1D-MS', 'CNN1D', 'Hybrid MST-Conformer']
summary_rows = []
for m_name in model_display_order:
    if m_name in all_results:
        df_m = all_results[m_name]
        summary_rows.append({
            'Model': m_name,
            'Bal. Acc.': f"{df_m['bal_acc'].mean():.3f} ± {df_m['bal_acc'].std():.3f}",
            'F1-Score': f"{df_m['f1'].mean():.3f} ± {df_m['f1'].std():.3f}",
            'AUC': f"{df_m['auc'].mean():.3f} ± {df_m['auc'].std():.3f}",
            'Accuracy': f"{df_m['accuracy'].mean():.3f} ± {df_m['accuracy'].std():.3f}",
            'Sensitivity': f"{df_m['sensitivity'].mean():.3f} ± {df_m['sensitivity'].std():.3f}",
            'Specificity': f"{df_m['specificity'].mean():.3f} ± {df_m['specificity'].std():.3f}",
        })

df_summary = pd.DataFrame(summary_rows)
print('\n' + '='*95)
print('TABLE 7 (REVISED): DEEP LEARNING PERFORMANCE COMPARISON UNDER 20-SECOND WINDOWS')
print('='*95)
print(df_summary.to_string(index=False))
print('='*95)

summary_csv = os.path.join(TARGET_DIR, 'table7_deep_learning_20s_summary.csv')
df_summary.to_csv(summary_csv, index=False)



Running baseline model: BiLSTM

STARTING 15-FOLD LOSO EVALUATION: BILSTM (20s Windows)
Partition Schedule: Identical to Ablation Study (v5 Ablation v2)
[01/15] Fold S2   (Val=S3  ) in 439.9s | BalAcc=0.459 | F1=0.025 | AUC=0.223 | Sens=0.015 | Spec=0.903
[02/15] Fold S3   (Val=S4  ) in 443.2s | BalAcc=0.649 | F1=0.545 | AUC=0.698 | Sens=0.562 | Spec=0.737
[03/15] Fold S4   (Val=S5  ) in 443.8s | BalAcc=0.505 | F1=0.300 | AUC=0.673 | Sens=0.264 | Spec=0.747
[04/15] Fold S5   (Val=S6  ) in 432.1s | BalAcc=0.547 | F1=0.493 | AUC=0.666 | Sens=0.740 | Spec=0.354
[05/15] Fold S6   (Val=S7  ) in 431.4s | BalAcc=0.585 | F1=0.409 | AUC=0.642 | Sens=0.343 | Spec=0.827
[06/15] Fold S7   (Val=S8  ) in 430.8s | BalAcc=0.615 | F1=0.533 | AUC=0.635 | Sens=0.693 | Spec=0.537


In [ ]:
# ==== BLOCK 9: STATISTICAL SIGNIFICANCE TESTING SUITE (PAIRED T-TEST, WILCOXON, COHEN'S D, BOOTSTRAP CI) ====
def run_paired_statistical_battery(hybrid_df, baseline_df, baseline_name, metrics=None):
    if metrics is None:
        metrics = ['bal_acc', 'f1', 'auc', 'accuracy', 'sensitivity', 'specificity']

    # Ensure subject-by-subject alignment
    merged = pd.merge(
        hybrid_df[['test_subject'] + metrics],
        baseline_df[['test_subject'] + metrics],
        on='test_subject', suffixes=('_hybrid', '_base')
    )

    stat_results = []
    for m in metrics:
        hyb = merged[f'{m}_hybrid'].values
        base = merged[f'{m}_base'].values
        diff = hyb - base
        n = len(diff)

        # 1. Paired Student's t-test
        t_stat, p_ttest = sp_stats.ttest_rel(hyb, base)

        # 2. Wilcoxon signed-rank test
        try:
            w_stat, p_wilcoxon = sp_stats.wilcoxon(hyb, base)
        except Exception:
            w_stat, p_wilcoxon = 0.0, 1.0

        # 3. Cohen's d for paired samples
        cohen_d = np.mean(diff) / (np.std(diff, ddof=1) + 1e-8)

        # 4. Bootstrap 95% Confidence Interval (10,000 resamples)
        np.random.seed(42)
        boot_means = [np.mean(np.random.choice(diff, size=n, replace=True)) for _ in range(10000)]
        ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

        stat_results.append({
            'Comparison': f'Hybrid vs {baseline_name}',
            'Metric': m,
            'Hybrid_Mean': np.mean(hyb),
            'Baseline_Mean': np.mean(base),
            'Mean_Diff': np.mean(diff),
            'Paired_t_p': p_ttest,
            'Wilcoxon_p': p_wilcoxon,
            'Cohens_d': cohen_d,
            'CI_95_Lower': ci_low,
            'CI_95_Upper': ci_high
        })

    return pd.DataFrame(stat_results)

df_hyb = all_results['Hybrid MST-Conformer']
stat_tables = []

for b_name in ['BiLSTM', 'CNN1D', 'CNN1D-MS']:
    df_base = all_results[b_name]
    df_stat_b = run_paired_statistical_battery(df_hyb, df_base, b_name)
    stat_tables.append(df_stat_b)

df_stats_master = pd.concat(stat_tables, ignore_index=True)
stats_csv = os.path.join(TARGET_DIR, 'statistical_significance_results_20s.csv')
df_stats_master.to_csv(stats_csv, index=False)

print('\n' + '='*105)
print('FORMAL STATISTICAL SIGNIFICANCE COMPARISON (15-FOLD LOSO PAIRS)')
print('='*105)
for comp in df_stats_master['Comparison'].unique():
    print(f"\n--- {comp} ---")
    sub = df_stats_master[df_stats_master['Comparison'] == comp]
    for _, r in sub.iterrows():
        sig_str = '***' if r['Paired_t_p'] < 0.01 else ('*' if r['Paired_t_p'] < 0.05 else 'ns')
        print(f"{r['Metric']:<12s}: Diff = {r['Mean_Diff']:+0.3f} | t-test p = {r['Paired_t_p']:.4f} ({sig_str:<3s}) | "
              f"Wilcoxon p = {r['Wilcoxon_p']:.4f} | Cohen's d = {r['Cohens_d']:+0.3f} | 95% CI: [{r['CI_95_Lower']:+0.3f}, {r['CI_95_Upper']:+0.3f}]")
print('='*105)



FORMAL STATISTICAL SIGNIFICANCE COMPARISON (15-FOLD LOSO PAIRS)

--- Hybrid vs BiLSTM ---
bal_acc     : Diff = +0.223 | t-test p = 0.0000 (***) | Wilcoxon p = 0.0001 | Cohen's d = +1.841 | 95% CI: [+0.164, +0.280]
f1          : Diff = +0.303 | t-test p = 0.0001 (***) | Wilcoxon p = 0.0001 | Cohen's d = +1.340 | 95% CI: [+0.199, +0.421]
auc         : Diff = +0.249 | t-test p = 0.0000 (***) | Wilcoxon p = 0.0001 | Cohen's d = +1.825 | 95% CI: [+0.181, +0.317]
accuracy    : Diff = +0.236 | t-test p = 0.0001 (***) | Wilcoxon p = 0.0009 | Cohen's d = +1.444 | 95% CI: [+0.151, +0.311]
sensitivity : Diff = +0.183 | t-test p = 0.1209 (ns ) | Wilcoxon p = 0.1688 | Cohen's d = +0.426 | 95% CI: [-0.023, +0.391]
specificity : Diff = +0.263 | t-test p = 0.0226 (*  ) | Wilcoxon p = 0.0084 | Cohen's d = +0.662 | 95% CI: [+0.061, +0.452]

--- Hybrid vs CNN1D ---
bal_acc     : Diff = +0.221 | t-test p = 0.0000 (***) | Wilcoxon p = 0.0001 | Cohen's d = +2.357 | 95% CI: [+0.175, +0.266]
f1          : Di